In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve
)

In [ ]:
from src.utils import load_object, read_csv_safely
from src.data_transformation import DataTransformation

model = load_object('../artifacts/model.pkl')
encoder = load_object('../artifacts/encoder.pkl')
scaler = load_object('../artifacts/scaler.pkl')
model

In [ ]:
test_df = read_csv_safely('../artifacts/test.csv')

transformer = DataTransformation(artifacts_dir='../artifacts')
transformer.encoder = encoder
transformer.scaler = scaler
transformer.numeric_columns = load_object('../artifacts/numeric_columns.pkl')
transformer.fitted = True

In [ ]:
X_test = transformer.transform(test_df.drop(columns=['is_claim']))
y_test = test_df['is_claim']

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
print('Test set size:', X_test.shape)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Claim', 'Claim'],
            yticklabels=['No Claim', 'Claim'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
print(classification_report(y_test, y_pred, target_names=['No Claim', 'Claim']))

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='#4C72B0', label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='grey')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_proba)

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, color='#C44E52')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.show()

In [ ]:
if hasattr(model, 'feature_importances_'):
    importances = pd.Series(model.feature_importances_, index=X_test.columns)
    top15 = importances.sort_values(ascending=False).head(15)
    plt.figure(figsize=(8, 6))
    sns.barplot(x=top15.values, y=top15.index, palette='mako')
    plt.title('Top 15 features - final model')
    plt.tight_layout()
    plt.show()
else:
    print('Selected model does not expose feature_importances_')

In [ ]:
sample_results = pd.DataFrame({
    'actual': y_test.values[:10],
    'predicted': y_pred[:10],
    'claim_probability': y_proba[:10].round(4),
})
sample_results